In [ ]:
import re
import random
import pandas as pd
from langdetect import detect
from tqdm import tqdm
from sklearn.utils import shuffle


# Data Loading

In [ ]:
# Load parallel subtitle data (English and French)
with open("OpenSubtitles.en-fr.en", encoding="utf-8") as f_en, \
     open("OpenSubtitles.en-fr.fr", encoding="utf-8") as f_fr:
    en_lines = f_en.readlines()
    fr_lines = f_fr.readlines()

# Randomly sample 10,000 sentence pairs for preprocessing
idx = random.sample(range(len(en_lines)), 10000)
df = pd.DataFrame({
    "src": [en_lines[i].strip() for i in idx],
    "tgt": [fr_lines[i].strip() for i in idx]
})

print(df.head())
print("Total number of samples:", len(df))


# Preprocessing

In [ ]:
def clean_text(
    text: str,
    remove_inside_parentheses: bool = True,
    to_lower: bool = True,
) -> str:
    """Clean and normalize subtitle text."""
    if not isinstance(text, str):
        return ""

    # 1. Remove HTML tags and entities
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"&[a-z]+;", "", text)

    # 2. Remove content inside parentheses (optional)
    if remove_inside_parentheses:
        text = re.sub(r"\(.*?\)", "", text)
        text = re.sub(r"\[.*?\]", "", text)
        text = re.sub(r"\{.*?\}", "", text)
    else:
        text = re.sub(r"[\(\)\[\]\{\}]", "", text)

    # 3. Remove timestamps and subtitle-specific notations
    text = re.sub(r"\d{2}:\d{2}:\d{2},\d{3}", "", text)
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"\(.*?\)", "", text)

    # 4. Trim non-word edges and normalize spaces
    text = re.sub(r"^\W+|\W+$", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    # 5. Lowercase (optional)
    if to_lower:
        text = text.lower()

    return text


def is_valid_sentence(text: str) -> bool:
    """Check if a sentence is valid and not noisy."""
    if not text or len(text.split()) < 2:
        return False
    if re.search(r"\d{2}:\d{2}:\d{2}", text):
        return False
    if re.match(r"^\[.*\]$", text):
        return False
    if re.match(r"^\d+$", text):
        return False
    if len(text) > 300:
        return False
    return True


def detect_language_safe(text, default="unk"):
    """Safely detect language (ignore exceptions)."""
    try:
        return detect(text)
    except:
        return default


In [ ]:
def preprocess_opensubs(df: pd.DataFrame, src_lang="en", tgt_lang="fr") -> pd.DataFrame:
    """Full preprocessing pipeline for OpenSubtitles dataset."""
    print("Step 1: Cleaning text ...")
    tqdm.pandas()
    df["src"] = df["src"].progress_apply(clean_text)
    df["tgt"] = df["tgt"].progress_apply(clean_text)

    print("Step 2: Removing invalid or noisy sentences ...")
    df = df[df["src"].apply(is_valid_sentence)]
    df = df[df["tgt"].apply(is_valid_sentence)]

    print("Step 3: Language detection (this may take some time) ...")
    df["src_lang"] = df["src"].progress_apply(detect_language_safe)
    df["tgt_lang"] = df["tgt"].progress_apply(detect_language_safe)
    df = df[(df["src_lang"] == src_lang) & (df["tgt_lang"] == tgt_lang)]

    print("Step 4: Filtering by length ...")
    df = df[df["src"].apply(lambda x: 3 <= len(x.split()) <= 80)]
    df = df[df["tgt"].apply(lambda x: 3 <= len(x.split()) <= 80)]

    print("Step 5: Removing duplicates ...")
    df = df.drop_duplicates(subset=["src", "tgt"])

    print("Step 6: Shuffling data ...")
    df = shuffle(df, random_state=42).reset_index(drop=True)

    print(f"Done. Remaining sentence pairs: {len(df):,}")
    return df


In [ ]:
clean_df = preprocess_opensubs(df, src_lang="en", tgt_lang="fr")


In [ ]:
output_path = "OpenSubtitles_en-fr_clean.csv"
clean_df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to {output_path}")
